In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import re
xlsx_file          = xlsx_file
PROJECT_ROOT = Path(
    os.getenv("PROJECT_ROOT", Path.cwd())
).resolve()

identifier = f'{P}P_{LoD}_Z{Z}_S2_{EXP_NAME}_{TIMESCALE}_{NOTEs}' 
identifier_idf = f"{P}P_{LoD}_Z{Z}_S2"

PROJECT_PARENT = PROJECT_ROOT.parent
POST_CAL_DIR = Path(f"eplus_diagnosis/idf-Oct/Z{Z}/auto-cal/{DATE}/base-idf")

idf_template = (
    PROJECT_PARENT
    / POST_CAL_DIR
    / f"{identifier_idf}.idf"
)

base_dir = Path(".").resolve()
exp_dir = base_dir / DATE / identifier

csv_path = (
    exp_dir
    / f"output_{EXP_NAME}_{TIMESCALE}"
    / "posterior_rand10_from_topfrac.csv"
)

pbook_path = Path(xlsx_file)

out_dir = exp_dir / "posterior_dir"
out_dir.mkdir(parents=True, exist_ok=True)

out_name_template = f"{identifier}_{{number}}"

NameError: name 'P' is not defined

In [ ]:
def load_and_select_posterior(
    csv_path: Path,
    out_csv: Path | None = None,
    target_n: int = 10,
) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    if df.empty:
        raise RuntimeError("Posterior CSV is empty")
    N = len(df)
    if N >= target_n:
        idx = np.linspace(0, N - 1, target_n)
        idx = np.round(idx).astype(int)
        idx = np.unique(idx)
        df = df.iloc[idx]
    else:
        print(f"⚠️  Only {N} rows available (< {target_n}); keeping all rows")
    df = df.reset_index(drop=True)
    if out_csv is not None:
        out_csv = Path(out_csv)
        out_csv.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out_csv, index=False)
        print(f"posterior written to: {out_csv.resolve()}")
    return df

def apply_shadow_rules(row_dict, calibrated_keys, shadow_rules):
    for (parent, child), func in shadow_rules.items():
        if parent in calibrated_keys and parent in row_dict:
            row_dict[child] = float(func(row_dict[parent]))
    return row_dict

def create_idf_files(
    template_file_name,
    cPath,
    variable_names,
    values,
    output_name_template,
    overwrite=False,
):
    if not overwrite:
        print("Overwrite disabled. Skipping IDF creation.")
        return []

    cPath = Path(cPath)
    cPath.mkdir(parents=True, exist_ok=True)

    template_text = Path(template_file_name).read_text(encoding="utf-8")
    file_paths = []

    for i, value_set in enumerate(values):
        f_out = template_text

        for var, val in zip(variable_names, value_set):
            placeholder = f"{{{{{var}}}}}"
            if placeholder in f_out and val is not None:
                f_out = f_out.replace(placeholder, f"{val:.6f}")

        out_name = output_name_template.format(number=f"{i:04d}")
        out_file = cPath / f"{out_name}.idf"
        out_file.write_text(f_out, encoding="utf-8")
        file_paths.append(out_file)

    return file_paths

for pth, label in [
    (idf_template, "Template IDF"),
    (csv_path, "Posterior CSV"),
    (pbook_path, "P-Book"),
]:
    if not pth.exists():
        raise FileNotFoundError(f"{label} not found: {pth}")

selected_csv = out_dir / "posterior_selected.csv"

df_post = load_and_select_posterior(
    csv_path,
    out_csv=selected_csv,
    target_n=40,
)

calibrated_params = df_post.columns.tolist()

df_pbook = pd.read_excel(pbook_path)

if not {"NAME", "Value-Ref"}.issubset(df_pbook.columns):
    raise RuntimeError("P-Book must contain columns: NAME, Value-Ref")

default_map = (
    df_pbook
    .set_index("NAME")["Value-Ref"]
    .apply(pd.to_numeric, errors="coerce")
    .to_dict()
)

template_text = idf_template.read_text(encoding="utf-8")

all_params = [
    p for p in default_map
    if f"{{{{{p}}}}}" in template_text
]

if "{{d1_htgsb_office}}" in template_text and "d1_htgsb_office" not in all_params:
    all_params.append("d1_htgsb_office")

unused = set(calibrated_params) - set(all_params)

shadow_rules = {
    ("d1_htgsp_office", "d1_htgsb_office"): lambda sp: sp - 6.0,
}

values = []

for _, row in df_post.iterrows():
    row_dict = {p: default_map.get(p) for p in all_params}

    for p in calibrated_params:
        if p in row_dict and pd.notna(row[p]):
            row_dict[p] = float(row[p])

    row_dict = apply_shadow_rules(row_dict, calibrated_params, shadow_rules)
    values.append([row_dict[p] for p in all_params])

idf_paths = create_idf_files(
    template_file_name=idf_template,
    cPath = base_dir.parent / f"{DATE}" / 'posterior' / f"{identifier_idf}_{NOTEs}",
    variable_names=all_params,
    values=values,
    output_name_template=out_name_template,
    overwrite=True,
)

print(f"✅ Generated {len(idf_paths)} IDF files in: {out_dir.resolve()}")

posterior written to: /Users/rui.bo/Desktop/Working/1-phd_mainworks/Y3/calibro_template/5P_bM_Z2_S2_FULL_M_sim/posterior_selected.csv
✅ Generated 10 IDF files in: /Users/rui.bo/Desktop/Working/1-phd_mainworks/Y3/calibro_template/5P_bM_Z2_S2_FULL_M_sim
